# QIAS RAG - Option C (Google Colab)

This notebook enables running the project on Google Colab using:
- **HuggingFace** backend (default and recommended)
- **Unsloth** backend (optional, GPU-only)

## Runtime requirements
- For `huggingface`: Colab GPU runtime recommended (T4 or better)
- For `unsloth`: Colab GPU runtime required (CUDA)

## What this notebook does
1. Mounts Google Drive
2. Clones or updates the repo
3. Installs dependencies
4. Sets backend to `huggingface` or `unsloth`
5. Builds/loads the knowledge base
6. Runs a smoke-test query
7. Optionally runs batch inference/evaluation

## Step 1: GPU Verification

In [1]:
# Verify GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Sat Mar  7 05:03:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             61W /  400W |    3100MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create directories in Drive
!mkdir -p /content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs
!mkdir -p /content/drive/MyDrive/QIAS26/qias_rag_thinking/data/vector_db
!mkdir -p /content/drive/MyDrive/QIAS26/qias_rag_thinking/results

print("✓ Google Drive mounted successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully


In [ ]:
# Backend selection (HF by default, Unsloth optional)
CLIENT_TYPE = "huggingface"  # choose: "huggingface" or "unsloth"

assert CLIENT_TYPE in {"huggingface", "unsloth"}
print(f"Selected backend: {CLIENT_TYPE}")

## Step 3: Clone Repositories

In [3]:
import os

REPO_URL = "https://github.com/swaileh/qias-mawarith-rag.git"
REPO_DIR = "/content/qias_rag_thinking"
DRIVE_REPO_DIR = "/content/drive/MyDrive/QIAS26/qias_rag_thinking"
SHARED_TASK_DIR = "/content/qias_shared_task_2026"

# Clone/update in Colab local disk for speed
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already exists in /content, pulling latest...")
    !git -C {REPO_DIR} pull

# Optional: if you keep your own copy in Drive, sync from there
if os.path.exists(DRIVE_REPO_DIR):
    print("Drive copy detected; using it as source of truth")
    !rsync -a --delete {DRIVE_REPO_DIR}/ {REPO_DIR}/

# Shared task dataset repo (optional but needed for batch eval cells)
if not os.path.exists(SHARED_TASK_DIR):
    !git clone https://github.com/QIAS/qias_shared_task_2026.git {SHARED_TASK_DIR}
else:
    !git -C {SHARED_TASK_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 4: Install Dependencies

In [4]:
# Base dependencies from project metadata
%cd /content/qias_rag_thinking
!pip install -q -U pip
!pip install -q -e ".[dev]"

# Optional GPU extras for better HF/Unsloth behavior
!pip install -q bitsandbytes accelerate

if CLIENT_TYPE == "unsloth":
    !pip install -q unsloth
    print("✓ Installed Unsloth backend dependencies")
else:
    print("✓ Using HuggingFace backend dependencies")

print("✓ Dependency setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00:00:0100:01
✓ Dependencies installed (HF mode for Qwen3.5)


## Step 5: Model Setup (HuggingFace Transformers)

In [5]:
# Using HuggingFace Transformers for Qwen3.5 (no Ollama needed)
print("="*80)
print("Qwen3.5 via HuggingFace Transformers")
print("="*80)
print("✓ No Ollama installation required")
print("✓ Model will be loaded directly from HuggingFace")
print("✓ Model: Qwen/Qwen3.5-9B-Instruct")
print("✓ Using 4-bit quantization (~6.6GB VRAM)")
print("="*80)

Qwen3.5 via HuggingFace Transformers
✓ No Ollama installation required
✓ Model will be loaded directly from HuggingFace
✓ Model: Qwen/Qwen3.5-9B-Instruct
✓ Using 4-bit quantization (~6.6GB VRAM)


## Step 6: Model Configuration

In [11]:
# Configure backend and generation settings
import yaml
from pathlib import Path

config_path = Path("/content/qias_rag_thinking/config/rag_config.yaml")
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["model"]["client_type"] = CLIENT_TYPE
config["model"]["hf_model_name"] = config["model"].get("hf_model_name", "Qwen/Qwen3.5-9B")
config["model"]["unsloth_model_name"] = config["model"].get("unsloth_model_name", "Qwen/Qwen3.5-9B")
config["model"]["unsloth_load_in_4bit"] = True
config["model"]["max_new_tokens"] = 1024
config["model"]["context_window"] = 8192

# Keep this disabled for faster runtime in smoke/batch tests
config.setdefault("evaluation", {})["enable_relevance_evaluation"] = False

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

print(f"✓ Updated config: model.client_type={CLIENT_TYPE}")
print(f"✓ Config file: {config_path}")

Configuring Qwen3.5 via HuggingFace Transformers...

Model Details:
  - Name: Qwen/Qwen3.5-9B-Instruct
  - Size: ~6.6GB (4-bit quantized)
  - Source: HuggingFace Hub

The model will download automatically when the pipeline initializes.
First download takes 5-10 minutes, then it's cached.

✓ Configuration saved for HuggingFace mode


In [13]:
# GPU and Memory Check for HuggingFace
import torch

print("=== GPU Check for HuggingFace Transformers ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU memory: {total_memory:.2f} GB")
    print(f"\n✓ Ready to load Qwen3.5 with 4-bit quantization (~6.6GB)")
    print(f"\nNext: Run the pipeline initialization cell below")
else:
    print("⚠ No GPU available - model loading will be very slow")

=== GPU Check for HuggingFace Transformers ===
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Total GPU memory: 85.09 GB

✓ Ready to load Qwen3.5 with 4-bit quantization (~6.6GB)

Next: Run the pipeline initialization cell below


## Step 7: Initialize RAG System

In [16]:
# Initialize the project pipeline
import os
import sys
import time

sys.path.insert(0, "/content/qias_rag_thinking")

from qias_mawarith_rag.pipeline import RAGPipeline

t0 = time.perf_counter()
pipeline = RAGPipeline(config_path="/content/qias_rag_thinking/config/rag_config.yaml")
init_s = time.perf_counter() - t0

print(f"✓ RAG Pipeline initialized in {init_s:.2f}s")
print(f"✓ Active backend: {pipeline.config['model']['client_type']}")
if pipeline.config['model']['client_type'] == 'unsloth':
    print("Note: Unsloth requires a CUDA GPU runtime in Colab")

✓ Copied RAG system from Drive
✓ Updated config to use Qwen/Qwen3.5-9B via HuggingFace (thinking enabled)
Initializing RAG Pipeline...
Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store initialized: 0 documents
Web search is disabled in configuration
Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

✓ Model loaded successfully
✓ Device: cuda:0
No documents in vector store for BM25 index
RAG Pipeline initialized successfully
✓ RAG Pipeline initialized


## Step 8: Build Knowledge Base from PDFs

In [17]:
# Build knowledge base from PDFs in Google Drive
pdf_directory = '/content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs'

# Check if PDFs exist
import os
pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files")

if pdf_files:
    # Build knowledge base
    pipeline.build_knowledge_base(pdf_directory)
    print("\n✓ Knowledge base built successfully")
    
    # Show stats
    stats = pipeline.vector_store.get_collection_stats()
    print(f"\nKnowledge Base Stats:")
    print(f"Total documents: {stats['total_documents']}")
    print(f"Sources: {stats['sample_sources']}")
else:
    print("⚠ No PDF files found. Please upload PDFs to:")
    print(f"  {pdf_directory}")

Found 3 PDF files

=== Building Knowledge Base ===
Found 3 PDF files in /content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs
Processed 03_matn_ar_rahabiyyah.pdf: 1 chunks from 14 pages
Processed 07_mukhtasar_al_wasit.pdf: 265 chunks from 54 pages
Processed arabic_talkhis_fiqh_al_fraid.pdf: 65 chunks from 19 pages
Total documents created: 331
Generating embeddings for 331 documents...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Added batch 1: 100 documents
Added batch 2: 100 documents
Added batch 3: 100 documents
Added batch 4: 31 documents
Total documents in collection: 331
Building BM25 index from vector store...
Built BM25 index with 331 documents
Knowledge base built with 331 document chunks

✓ Knowledge base built successfully

Knowledge Base Stats:
Total documents: 331
Sources: ['07_mukhtasar_al_wasit.pdf', '03_matn_ar_rahabiyyah.pdf']


In [22]:
# Quick HuggingFace verification
print("=== HuggingFace Setup Verification ===")
try:
    import transformers
    print(f"✓ Transformers: {transformers.__version__}")
    print("✓ Ready for Qwen/Qwen3.5-9B")
except ImportError:
    print("✗ transformers not installed")

=== HuggingFace Setup Verification ===
✓ Transformers: 5.2.0
✓ Ready for Qwen/Qwen3.5-9B


## Step 9: Test with Single Question

In [23]:
# Test with example question - SIMPLIFIED TEST FIRST
test_question = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"

print(f"Question: {test_question}")
print(f"Model: {pipeline.qwen_client.model_name}\n")

# First, try a simple direct test to verify model works
print("=== Pre-test: Simple model check ===")
simple_test = pipeline.qwen_client.generate("Say 'Working' in Arabic.", max_tokens=50)
print(f"Simple test result: '{simple_test}'\n")

if not simple_test or len(simple_test.strip()) == 0:
    print("✗ Model is not responding. Possible issues:")
    print("  1. Model not fully loaded - try waiting a bit longer")
    print("  2. GPU memory issue - try a smaller model")
    print("  3. HuggingFace access issue - check internet or token")
else:
    print("✓ Model is working. Running full RAG query...\n")
    
    # Run RAG pipeline
    result = pipeline.query(test_question, top_k=5)
    
    # Check for errors
    if result.get('error'):
        print(f"\n✗ Error during query: {result['error']}")
    
    # 1. Display the thinking process
    print("\n" + "="*80)
    print("THINKING PROCESS (first 500 chars):")
    print("="*80)
    thinking = result['parsed_output'].get('thinking', '')
    if thinking:
        print(thinking[:500] + "..." if len(thinking) > 500 else thinking)
    else:
        print("No thinking output captured")
        if result['raw_output']:
            print(f"\nDebug - Raw output preview: {result['raw_output'][:200]}...")

    # 2. Show the structured JSON output
    print("\n" + "="*80)
    print("STRUCTURED OUTPUT:")
    print("="*80)
    import json
    structured = result['parsed_output'].get('structured_output')
    if structured:
        print(json.dumps(structured, ensure_ascii=False, indent=2))
    else:
        print("No structured output available")
        raw = result.get('raw_output', '')
        if raw:
            print(f"Raw output ({len(raw)} chars): {raw[:500]}...")
        else:
            print("Raw output is empty")

    # 3. Print quality metrics
    print("\n" + "="*80)
    print("QUALITY METRICS:")
    print("="*80)
    print(f"Parsing Success: {result['parsed_output'].get('parsing_success', False)}")
    print(f"Validation Success: {result['parsed_output'].get('validation_success', False)}")

    if result.get('thinking_quality'):
        print(f"\nThinking Quality Scores:")
        print(f"  - Quality Score:     {result['thinking_quality'].get('quality_score', 0):.2f}")
        print(f"  - Terminology Score: {result['thinking_quality'].get('terminology_score', 0):.2f}")
        print(f"  - Completeness Score: {result['thinking_quality'].get('completeness_score', 0):.2f}")
    else:
        print("  (No thinking quality metrics available)")

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"✓ Model used: {pipeline.qwen_client.model_name}")
    print(f"✓ Retrieved docs: {len(result['retrieved_docs'])}")
    print(f"✓ Response length: {len(result['raw_output'])} chars")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟
Model: Qwen/Qwen3.5-9B

=== Pre-test: Simple model check ===
Simple test result: 'Thinking Process:

1.  **Analyze the Request:** The user is asking for the Arabic translation of the English word "Working".

2.  **Identify the Context:** The user's prompt includes a system instruction: "You are'

✓ Model is working. Running full RAG query...


=== Processing Question ===
مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟...
Retrieving documents...
Reranking documents...
Building prompt...
Generating response with Qwen3...
Parsing output...
✓ Parsing success: True
✓ Validation success: False

THINKING PROCESS (first 500 chars):
No thinking output captured

Debug - Raw output preview: The user is asking me to solve an Islamic inheritance problem based on the provided context.

The question is: "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟" (He died and left: Mother, Father, Daughte...

STRUCTURED OUTPUT:
{
  "heirs": [
    {
      "heir": 

In [ ]:
# Optional: latency smoke benchmark
import time

question = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"

t0 = time.perf_counter()
result = pipeline.query(question, top_k=5)
query_s = time.perf_counter() - t0

print(f"Latency: {query_s:.2f}s")
print(f"Parsing success: {result['parsed_output'].get('parsing_success')}")
print(f"Validation success: {result['parsed_output'].get('validation_success')}")
print(f"Retrieved docs: {len(result.get('retrieved_docs', []))}")

## Step 10: Run Batch Evaluation

In [ ]:
# Load QIAS dataset (with fallback synthetic examples)
import json
import os

dataset_file = '/content/qias_shared_task_2026/data/jsonl/mawarith/qias2025_almawarith_part1.json'

if os.path.exists(dataset_file):
    with open(dataset_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print(f"Loaded {len(dataset)} questions from dataset")
else:
    print("Dataset file not found, using synthetic fallback cases")
    dataset = [
        {
            "id": f"synthetic_{i}",
            "question": "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟",
            "output": {},
        }
        for i in range(10)
    ]

num_questions = min(10, len(dataset))
test_subset = dataset[:num_questions]

print(f"\nProcessing {len(test_subset)} questions...")
results = pipeline.batch_query(test_subset, save_results=True)

successful = sum(1 for r in results if r['parsed_output'].get('validation_success'))
print(f"\n✓ Successfully processed: {successful}/{len(results)}")

## Step 11: Evaluate with MIR-E

In [ ]:
# Run MIR-E evaluation (only if real reference file exists)
import os
from qias_mawarith_rag.evaluation.mir_e_wrapper import MIREvaluator

predictions_file = '/content/qias_rag_thinking/results/rag_predictions.json'
reference_file = dataset_file

if os.path.exists(reference_file) and os.path.exists(predictions_file):
    evaluator = MIREvaluator()
    results = evaluator.evaluate_predictions(predictions_file, reference_file)

    print("\n" + "="*80)
    print("MIR-E EVALUATION RESULTS")
    print("="*80)
    print(f"\nOverall MIR-E Score: {results.get('average_mir_e', 0):.4f}")
    print(f"Total Cases: {results.get('total_cases', 0)}")

    print("\nSubscores:")
    for key, value in results.get('subscores', {}).items():
        print(f"  {key}: {value:.4f}")

    evaluator.generate_report(results, 'evaluation_report.txt')
else:
    print("Skipping MIR-E evaluation because dataset/predictions file is missing.")

## Step 12: Visualize Results

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Create DataFrame from results
if 'per_case' in results:
    df = pd.DataFrame(results['per_case'])
    
    # Score distribution
    fig = px.histogram(df, x='mir_e', nbins=20, 
                       title='MIR-E Score Distribution',
                       labels={'mir_e': 'MIR-E Score', 'count': 'Number of Cases'})
    fig.show()
    
    # Subscore comparison
    subscores = results.get('subscores', {})
    fig = go.Figure(data=[
        go.Bar(x=list(subscores.keys()), y=list(subscores.values()))
    ])
    fig.update_layout(title='Subscore Performance',
                      xaxis_title='Metric',
                      yaxis_title='Score',
                      yaxis_range=[0, 1])
    fig.show()
    
    print(f"\nScore Statistics:")
    print(df['mir_e'].describe())

## Step 13: Save Results to Drive

In [ ]:
# Copy results to Google Drive
results_dir = '/content/drive/MyDrive/QIAS_RAG/results'

!cp -r /content/qias_rag_thinking/results/* {results_dir}/

print(f"✓ Results saved to Google Drive: {results_dir}")
print("\nFiles saved:")
!ls -lh {results_dir}/

## Next Steps

1. **Improve Performance**: If MIR-E score < 99%, analyze errors and refine
2. **Add More PDFs**: Upload additional Islamic law books to knowledge base
3. **Enable Web Search**: Configure Tavily API in config for web search augmentation
4. **Run Full Evaluation**: Process entire dataset (200+ cases)
5. **Fine-tune**: Optionally fine-tune Qwen3 on QIAS dataset

## Troubleshooting

- **GPU Memory Error**: Reduce batch size or use smaller model (qwen3:7b-thinking)
- **Model Loading Error**: Check HuggingFace token and internet connection
- **Low Scores**: Check retrieved context relevance, add more PDFs, adjust retrieval parameters

## Contact

For questions about the QIAS 2026 shared task, visit: https://sites.google.com/view/qias2026